<a href="https://colab.research.google.com/github/arinjay-singh/econ3916-statistical-machine-learning/blob/main/Class%2013%20/%20class13_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)
df.head()

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54


In [5]:
naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        19:56:16   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [6]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        19:56:21   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [9]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802139


In [16]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ─────────────────────────────────────────────
# 1. Synthetic Data (mirrors your lab setup)
# ─────────────────────────────────────────────
np.random.seed(42)
n = 200

property_age = np.random.uniform(1, 50, n)          # years
distance_to_tech_hub = np.random.uniform(0.5, 20, n) # miles

# True DGP: Sale_Price = 500000 - 3000*Age - 8000*Distance + noise
sale_price = (
    500_000
    - 3_000 * property_age
    - 8_000 * distance_to_tech_hub
    + np.random.normal(0, 25_000, n)
)

df = pd.DataFrame({
    "Sale_Price": sale_price,
    "Property_Age": property_age,
    "Distance_to_Tech_Hub": distance_to_tech_hub,
})

# ─────────────────────────────────────────────
# 2. OLS Regression via statsmodels
# ─────────────────────────────────────────────
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])
y = df["Sale_Price"]

model = sm.OLS(y, X).fit()
print(model.summary())

# ─────────────────────────────────────────────
# 3. Extract Coefficients from statsmodels Results
#
#    model.params is a pandas Series indexed by variable name:
#      model.params["const"]                  → β₀ (intercept)
#      model.params["Property_Age"]           → β₁ (slope for age)
#      model.params["Distance_to_Tech_Hub"]   → β₂ (slope for distance)
#
#    These are the OLS point estimates (β-hat) that minimize
#    the sum of squared residuals.
# ─────────────────────────────────────────────
beta_0 = model.params["const"]               # intercept
beta_1 = model.params["Property_Age"]        # coefficient for Property_Age
beta_2 = model.params["Distance_to_Tech_Hub"] # coefficient for Distance_to_Tech_Hub

print(f"\nExtracted Coefficients:")
print(f"  β₀ (Intercept)             : {beta_0:,.2f}")
print(f"  β₁ (Property_Age)          : {beta_1:,.2f}")
print(f"  β₂ (Distance_to_Tech_Hub)  : {beta_2:,.2f}")

# ─────────────────────────────────────────────
# 4. Build Meshgrid for the Regression Surface
#
#    np.linspace creates 50 evenly spaced values spanning
#    each predictor's observed range — this is the grid of
#    (X1, X2) combinations we'll evaluate the plane over.
#
#    np.meshgrid then takes those two 1-D arrays and returns
#    two 2-D arrays (age_grid, dist_grid) where every cell
#    (i, j) holds one (X1, X2) pair — effectively a Cartesian
#    product of all X1 values × all X2 values.
#
#    The predicted surface is then simply the linear equation
#    evaluated at every grid point:
#        Z[i,j] = β₀ + β₁ * age_grid[i,j] + β₂ * dist_grid[i,j]
#
#    Because OLS produces a linear model, this surface is a
#    flat plane in 3-D space — no curvature.
# ─────────────────────────────────────────────
n_grid = 50  # resolution of the surface (50×50 = 2,500 grid points)

# 1-D arrays spanning the observed range of each predictor
age_range  = np.linspace(df["Property_Age"].min(),
                         df["Property_Age"].max(), n_grid)
dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(),
                         df["Distance_to_Tech_Hub"].max(), n_grid)

# 2-D grids: each is shape (n_grid, n_grid)
#   age_grid[i, j]  = age_range[j]   (age varies across columns)
#   dist_grid[i, j] = dist_range[i]  (distance varies across rows)
age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# Evaluate the fitted plane at every (age, distance) grid point
price_surface = beta_0 + beta_1 * age_grid + beta_2 * dist_grid

# ─────────────────────────────────────────────
# 5. Compute Residuals (color-code scatter points)
#    Points above the plane → positive residual (under-predicted)
#    Points below the plane → negative residual (over-predicted)
# ─────────────────────────────────────────────
y_hat = model.fittedvalues
residuals = y - y_hat  # actual − predicted

# ─────────────────────────────────────────────
# 6. Build the Plotly 3-D Figure
# ─────────────────────────────────────────────
fig = go.Figure()

# --- 6a. Regression Surface (hyperplane) ---
fig.add_trace(go.Surface(
    x=age_grid,           # 2-D array: Property_Age values across the grid
    y=dist_grid,          # 2-D array: Distance_to_Tech_Hub values across the grid
    z=price_surface,      # 2-D array: predicted Sale_Price at each grid point
    colorscale="Blues",
    opacity=0.55,
    showscale=False,
    name="Regression Hyperplane",
    hovertemplate=(
        "Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} mi<br>"
        "Predicted Price: $%{z:,.0f}<extra>Hyperplane</extra>"
    ),
))

# --- 6b. Scatter: Data Points (colored by residual magnitude) ---
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    marker=dict(
        size=4,
        color=residuals,              # residual value drives color
        colorscale="RdBu",            # red = over-predicted, blue = under-predicted
        cmin=-residuals.abs().max(),  # symmetric around 0 so mid-range = white
        cmax= residuals.abs().max(),
        colorbar=dict(
            title="Residual ($)",
            thickness=14,
            len=0.6,
            x=1.02,
        ),
        opacity=0.85,
    ),
    name="Observed Sales",
    hovertemplate=(
        "Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} mi<br>"
        "Actual Price: $%{z:,.0f}<extra>Observed</extra>"
    ),
))

# --- 6c. Vertical residual lines connecting each point to the plane ---
for i in range(len(df)):
    fig.add_trace(go.Scatter3d(
        x=[df["Property_Age"].iloc[i],   df["Property_Age"].iloc[i]],
        y=[df["Distance_to_Tech_Hub"].iloc[i], df["Distance_to_Tech_Hub"].iloc[i]],
        z=[df["Sale_Price"].iloc[i],      y_hat.iloc[i]],
        mode="lines",
        line=dict(color="rgba(120,120,120,0.18)", width=1),
        showlegend=False,
        hoverinfo="skip",
    ))

# ─────────────────────────────────────────────
# 7. Layout & Annotations
# ─────────────────────────────────────────────
r2   = model.rsquared
rmse = np.sqrt(model.mse_resid)

fig.update_layout(
    title=dict(
        text=(
            f"OLS Regression Hyperplane — Sale Price on Property Age & Distance to Tech Hub<br>"
            f"<sup>β₀={beta_0:,.0f} | β₁(Age)={beta_1:,.0f} | β₂(Dist)={beta_2:,.0f} "
            f"| R²={r2:.3f} | RMSE=${rmse:,.0f}</sup>"
        ),
        x=0.5,
        font=dict(size=14),
    ),
    scene=dict(
        xaxis=dict(title="Property Age (years)", backgroundcolor="rgb(245,245,255)"),
        yaxis=dict(title="Distance to Tech Hub (miles)", backgroundcolor="rgb(245,255,245)"),
        zaxis=dict(title="Sale Price ($)", backgroundcolor="rgb(255,250,245)"),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9)),
        aspectratio=dict(x=1.2, y=1.2, z=0.9),
    ),
    legend=dict(x=0.01, y=0.95, bgcolor="rgba(255,255,255,0.7)"),
    margin=dict(l=0, r=80, t=90, b=0),
    width=1000,
    height=700,
)

import plotly.io as pio
pio.renderers.default = "colab"
fig.show()

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.867
Model:                            OLS   Adj. R-squared:                  0.866
Method:                 Least Squares   F-statistic:                     643.9
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           3.93e-87
Time:                        20:08:58   Log-Likelihood:                -2305.9
No. Observations:                 200   AIC:                             4618.
Df Residuals:                     197   BIC:                             4628.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 5.022e+05 